# Laptop performance investigation -- running notebook

Companion notebook to `Jacobik — Project State & Performance Handoff.md`
(2026-09-18). That handoff's "real anomaly": production settings
(`beam_width=16, n_coord_samples=8`) measure 68.79s / 88.19s / 83.90s on this
project's own 2-core cloud CPU sandbox (identical current code,
`use_symmetry_enumeration=True`, CPU-only), but the owner's laptop reportedly
takes 7+ minutes on the same CN9 case with the same code, CPU-only on both
machines. The architecture change (screen-then-resolve -> full-batch) only
explains a ~1.5-2x slowdown from the pre-batching baseline (~48-49s) -- not
the ~5-6x gap between the cloud-sandbox number and the laptop number. That
remaining gap is the actual thing this notebook chases.

This is a new notebook (kept separate from `site_enumeration_experiments.ipynb`,
which is about the site-enumeration/dedup feature work, not this performance
investigation) so this effort's log doesn't get mixed into that one. Same
append-only convention as the other notebooks in this project: each dated
section below is a step actually run, with real code and real output, new
work appended at the bottom, existing sections left alone.

## Setup

In [1]:
from __future__ import annotations

import sys, os
os.environ['KMP_DUPLICATE_LIB_OK'] = "True"
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "jacobik").exists():
            return candidate
    raise RuntimeError(
        "Could not find the Jacobik project root (looked for pyproject.toml "
        "+ src/jacobik above the notebook's directory). Open this notebook "
        "from inside the project tree."
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\alanc\sus folder\CCCSEF\Jacobik v3


## 2026-09-18 -- The CN9 reproduction case

Same case and settings used for the cloud-sandbox numbers in the handoff doc
and in `site_enumeration_experiments.ipynb`'s cell producing `r_prod`
(`[La(EDTA)(H2O)3]-`, production `beam_width=16, n_coord_samples=8`),
redefined here so this notebook is self-contained and can be run on its own,
in order, on the laptop.

In [2]:
import logging

from jacobik.engine import ComplexBuilder, LigandInput

logging.getLogger("jacobik").setLevel(logging.WARNING)

EDTA_SMILES = "[O-]C(=O)CN(CC(=O)[O-])CCN(CC(=O)[O-])CC(=O)[O-]"
edta_donors = [[4], [11], [0], [8], [15], [19]]


def make_la_edta_3h2o_cn9():
    edta_ligand = LigandInput(EDTA_SMILES, donors=edta_donors)
    waters = [LigandInput("O", donors=[[0]]) for _ in range(3)]
    return ComplexBuilder(
        metal="La", geometry="tricap_pris", charge=-1, spin=1, ligands=[edta_ligand] + waters
    )


DEFAULT_SOLVE_KWARGS = dict(beam_width=16, n_ik_steps=200, lr=0.1, n_coord_samples=8, seed=42)

b = make_la_edta_3h2o_cn9()
print(f"builder constructed OK, {len(b.ligands)} ligand(s)")

builder constructed OK, 4 ligand(s)


### Environment / threading / BLAS diagnostics

**Run this on the laptop, inside `jacobik-env`.** I could not run this cell
myself and produce real output for it: the device bridge's shell tool (used
to write this notebook) runs in an isolated sandbox VM with no
`jacobik-env`/`torch` installed -- it is not this laptop's real Windows
environment, so anything measured there would not be a laptop number.
Computer-use access to a terminal on this device is also click-only for
safety (no typing), so driving a real shell that way isn't possible either.

One concrete thing to look at once this runs: `pyproject.toml` pins plain
`torch` with no MKL/version constraint, and this device reports an arm64
host -- if `jacobik-env`'s torch build lacks a tuned BLAS backend for this
CPU (or is running under x86 emulation), `torch.__config__.show()` /
`torch.__config__.parallel_info()` below should show it directly.

In [3]:
import os
import platform

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
print(f"platform: {platform.platform()}")
print(f"processor: {platform.processor()}")
print(f"logical cpu count (os.cpu_count): {os.cpu_count()}")
try:
    import psutil
    print(f"physical cores (psutil): {psutil.cpu_count(logical=False)}")
    print(f"logical cores (psutil): {psutil.cpu_count(logical=True)}")
except ImportError:
    print("psutil not installed -- `pip install psutil` for a physical-core count")

for var in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "KMP_AFFINITY",
):
    print(f"env {var} = {os.environ.get(var, '<unset>')}")

import torch

print()
print(f"torch version: {torch.__version__}")
print(f"torch.get_num_threads(): {torch.get_num_threads()}")
print(f"torch.get_num_interop_threads(): {torch.get_num_interop_threads()}")
print(f"torch.backends.mkl.is_available(): {torch.backends.mkl.is_available()}")
print(f"torch.backends.mkldnn.is_available(): {torch.backends.mkldnn.is_available()}")
print(f"torch.backends.openmp.is_available(): {torch.backends.openmp.is_available()}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print()
print("torch.__config__.show():")
print(torch.__config__.show())
print("torch.__config__.parallel_info():")
print(torch.__config__.parallel_info())

ENVIRONMENT
platform: Windows-10-10.0.26200-SP0
processor: ARMv8 (64-bit) Family 8 Model 1 Revision 201, Qualcomm Technologies Inc
logical cpu count (os.cpu_count): 12
physical cores (psutil): 12
logical cores (psutil): 12
env OMP_NUM_THREADS = <unset>
env MKL_NUM_THREADS = <unset>
env OPENBLAS_NUM_THREADS = <unset>
env NUMEXPR_NUM_THREADS = <unset>
env KMP_AFFINITY = <unset>

torch version: 2.11.0+cpu
torch.get_num_threads(): 12
torch.get_num_interop_threads(): 12
torch.backends.mkl.is_available(): True
torch.backends.mkldnn.is_available(): True
torch.backends.openmp.is_available(): True
torch.cuda.is_available(): False

torch.__config__.show():
PyTorch built with:
  - C++ Version: 201703
  - MSVC 194234444
  - Intel(R) oneAPI Math Kernel Library Version 2025.3-Product Build 20251007 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.10.2 (Git Hash f1d471933dc852f956fd05389f9313c7148783d5)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capabilit

### Reproduce the CN9 case at production settings, with cProfile

First a plain timing run (directly comparable to the cloud-sandbox
`elapsed=` numbers: 68.79s / 88.19s / 83.90s), then a `cProfile` run
(directly comparable to the cloud-sandbox profile: `_run_backward` dominant
at 39.7s of 68.6s total) so the laptop's own top cost centers can be read off
and compared line-for-line. Also run on the laptop, for the same reason as
the diagnostics cell above.

In [1]:
import cProfile
import io
import pstats
import time

t0 = time.time()
r_laptop = make_la_edta_3h2o_cn9().solve(
    **DEFAULT_SOLVE_KWARGS, n_conformers=10, fast_embed=False, use_symmetry_enumeration=True,
)
elapsed = time.time() - t0
print(
    f"elapsed={elapsed:.2f}s  n_conformers={len(r_laptop.conformers)}  "
    f"n_candidates={r_laptop.assignment_search['n_candidates']}  "
    f"best_score={max(c.score for c in r_laptop.conformers):.4f}"
)

pr = cProfile.Profile()
pr.enable()
t0 = time.time()
r_laptop_prof = make_la_edta_3h2o_cn9().solve(
    **DEFAULT_SOLVE_KWARGS, n_conformers=10, fast_embed=False, use_symmetry_enumeration=True,
)
prof_elapsed = time.time() - t0
pr.disable()
print(f"\nprofiled elapsed={prof_elapsed:.2f}s")

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats("tottime")
ps.print_stats(25)
print(s.getvalue())

elapsed=292.30s  n_conformers=10  n_candidates=20  best_score=-4.5255

profiled elapsed=345.69s
         3177821 function calls (3015762 primitive calls) in 345.544 seconds

   Ordered by: internal time
   List reduced from 730 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      200  189.613    0.948  189.613    0.948 {method 'run_backward' of 'torch._C._EngineBase' objects}
     1417   22.142    0.016   22.142    0.016 {built-in method torch.logsumexp}
     3246   14.358    0.004   24.380    0.008 C:\Users\alanc\sus folder\CCCSEF\Jacobik v3\src\jacobik\ik\kernels.py:41(rodrigues)
    43231   11.502    0.000   11.502    0.000 {method 'sum' of 'torch._C.TensorBase' objects}
      201   10.169    0.051   13.418    0.067 C:\Users\alanc\sus folder\CCCSEF\Jacobik v3\src\jacobik\ik\geometry.py:782(q6_contacts)
    13599    6.474    0.000    6.474    0.000 {built-in method torch._C._linalg.linalg_cross}
    11099    6.026    0.001   11.

## 2026-09-18 (cont'd) -- Real laptop numbers, and what they show

The two cells above were run on the laptop, in `jacobik-env` (diagnostics
cell output not yet captured here -- see the ask at the bottom). Results:

| metric | cloud sandbox (3 runs) | laptop | ratio |
|---|---|---|---|
| plain `elapsed=` | 68.79s / 88.19s / 83.90s | **292.30s** | 3.3-4.3x |
| `cProfile`-instrumented elapsed | 68.56s (cell 76 above) | **345.54s** | 5.04x |
| `best_score` | -4.53 / -4.5342 | -4.5255 | consistent (~-4.5, matches the regression-check quality floor) |
| `n_candidates` | 20 | 20 | consistent |

So: this is real, reproducible, and correctness is intact (same
`n_candidates`, same `best_score` ballpark) -- this is a pure performance
gap, not a correctness regression. The plain-run ratio (3.3-4.3x) is a bit
under the "7+ minutes" originally reported (which would be ~5-6x), but
that number was a rough impression, not a controlled measurement, and
292s/4.87min against an 80s-ish cloud baseline is already a large,
precisely reproduced gap worth explaining on its own -- it doesn't need to
hit exactly 7 minutes to be the same phenomenon.

### Function-by-function: the slowdown is NOT uniform

Comparing `tottime` for every function that appears in both profiles (cloud
cell 76 above, `List reduced from 1031 to 15`; laptop cell above, `List
reduced from 730 to 25`):

| function | cloud tottime | laptop tottime | ratio | calls |
|---|---|---|---|---|
| `q6_contacts` | 5.931s | 10.169s | **1.7x** | 201 |
| `linalg_norm` | 1.342s | 3.558s | **2.7x** | 603 |
| `linalg_svd` | 1.205s | 4.652s | **3.9x** | 2,760 |
| `run_backward` | 39.659s | 189.613s | **4.8x** | 200 |
| `logsumexp` | 4.324s | 22.142s | **5.1x** | 1,417 |
| `_apply_torsion` | 0.986s | 5.911s | **6.0x** | 3,246 |
| `_apply_base_transform` | 0.617s | 4.000s | **6.5x** | 859 |
| `linalg_cross` | 0.938s | 6.474s | **6.9x** | 13,599 |
| `_matrix_to_axisangle` | 0.513s | 3.563s | **6.9x** | 10,240 |
| `axisangle_to_matrix` | 0.889s | 6.026s | **6.8x** | 11,099 |
| `kabsch_init` | 0.576s | 4.025s | **7.0x** | 4 |
| `rodrigues` | 1.262s | 14.358s | **11.4x** | 3,246 |
| `sum` | 0.949s | 11.502s | **12.1x** | 43,231 |

**This is the actual finding.** If the laptop just had a slower CPU or a
slower BLAS build across the board, every row would sit near the same
ratio (roughly the overall 5.0x). It doesn't. The two heaviest, most
fused, least-frequently-called operations -- `run_backward` (200 calls,
each one big) and `linalg_svd`/`linalg_norm` (a few thousand calls each
doing real batched linear algebra on B=2560) -- sit at 2.7-4.8x, close to
the overall average. But the functions built from many small, individually
cheap tensor ops -- `rodrigues` and `sum` above all, but also
`linalg_cross`, `axisangle_to_matrix`, `_matrix_to_axisangle`,
`kabsch_init`, `_apply_base_transform` -- sit at 6-12x, roughly double the
average slowdown, and `q6_contacts` (comparatively few, larger calls) sits
at only 1.7x, the least-slowed function in the whole table.

That shape -- many-tiny-calls functions disproportionately slower,
few-big-calls functions closer to the baseline slowdown -- is the
signature of **per-call dispatch/thread-pool overhead**, not raw compute
throughput. `sum` alone is called 43,231 times; `rodrigues` and its
callees fire 3,246 times per solve but each does several small
tensor ops. If every individual torch op pays a roughly fixed
launch/teardown cost (thread-pool wake-up, OpenMP/MKL thread-pool
handling, which is historically worse on Windows than Linux), that fixed
cost barely dents `run_backward`'s 200 big calls but compounds hard across
tens of thousands of small ones -- exactly this pattern. This is
suggestion #2 from the handoff doc's investigation list
(`torch.get_num_threads()` / thread-thrashing), now backed by a concrete
profile shape rather than just a hypothesis.

**Still needed to confirm this:** the diagnostics cell above (thread
counts, `torch.__config__.show()`, BLAS backend, physical core count) --
its output hasn't landed in this notebook yet. That's the piece that turns
"the profile shape is consistent with thread-pool overhead" into "here is
the actual thread/BLAS misconfiguration and here's the fix" (e.g. an
explicit `torch.set_num_threads()` matched to physical cores, or an MKL/OpenMP
env var). Please run that cell too and re-save.